<a href="https://colab.research.google.com/github/SY-256/llms-from-scratch/blob/main/notebooks/Appendix_D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 訓練ループに高度なテクニックと追加する
- 学習ウォームアップ
- コサイン減衰
- 勾配クリッピング

In [ ]:
! git clone https://github.com/rasbt/LLMs-from-scratch.git

In [ ]:
import sys
sys.path.append('/content/LLMs-from-scratch/appendix-D/01_main-chapter-code')

In [ ]:
# 訓練モデルを初期化
import torch
from previous_chapters import GPTModel

GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.to(device)
model.eval();

In [ ]:
# データローダーの初期化
import os
import requests
import urllib.request

file_path = "./the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

if not os.path.exists(file_path):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    text_data = response.text
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()


In [ ]:
# text_dataをデータローダーに読み込む
from previous_chapters import create_dataloader_v1

train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))

torch.manual_seed(123)

train_loader = create_dataloader_v1(
    text_data[:split_idx],
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0,
)

val_loader = create_dataloader_v1(
    text_data[split_idx:],
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=False,
    num_workers=0,
)

## D.1 学習率ウォームアップ
- モデルの訓練を安定化させることができる
- 学習率の初期値(`initial_lr`)をかなり低く設定し、そこからユーザー指定の最大値(`peak_lr`)まで徐々に引き上げていく
- より小さな重みで訓練を開始すると、訓練段階で安定性を損なうような大きな更新にモデルが遭遇するリスクを減らすことができる

In [ ]:
n_epochs = 15
initial_lr = 0.0001
peak_lr = 0.01

- ウォームアップのステップ数は、通常は全ステップ数の0.1 ~ 20%の間で設定する

In [ ]:
# ステップ数の計算
total_steps = len(train_loader) * n_epochs
warmup_steps = int(0.2 * total_steps) # 20%のウォームアップ
print(warmup_steps)

- 最初の27回の訓練ステップで学習率を初期値の0.0001から0.01に引き上げるには、20回のウォームアップステップが必要

In [ ]:
# 訓練ループ
optimizer = torch.optim.AdamW(model.parameters(), weight_decay=0.1)

# 20回ウォームアップステップごとにinitial_lrをどれくらい増やすかよって決まる
lr_increment = (peak_lr - initial_lr) / warmup_steps

global_step = -1
track_lrs = []

for epoch in range(n_epochs):
    for input_batch, target_batch in train_loader:
        optimizer.zero_grad()
        global_step += 1

        if global_step < warmup_steps:
            # まだウォームアップ段階であれば学習率を更新
            lr = initial_lr + global_step * lr_increment
        else:
            lr = peak_lr

        for param_group in optimizer.param_groups:
            # 計算した学習率をオプティマイザに適用
            param_group["lr"] = lr

        track_lrs.append(optimizer.param_groups[0]["lr"])

In [ ]:
# 学習率のウォームアップが正しく機能しているか確認
import matplotlib.pyplot as plt

plt.figure(figsize=(5, 3))
plt.xlabel("Steps")
plt.ylabel("Learning rate")
total_training_steps = len(train_loader) * n_epochs
plt.plot(range(total_training_steps), track_lrs)
plt.tight_layout()
plt.show()

- 学習率が低い値から始まり、最大値に達するまでに20ステップにわたって上昇していることがわかる

## D.2 コサイン減衰
- 訓練エポックを通じて学習率を調整し、ウォームアップ後はコサイン曲線に従わせる
- よく使用されるコサイン減衰の一種に、学習率をほぼゼロまで減衰させ、コサイン曲線の半周期分の軌跡を模倣させるものがある
- コサイン減衰における学習率の逓減は、モデルが重みを更新するペースを減少させることを目的としている
- コサイン減衰が特に重要なのは、訓練中に損失の極小値をオーバーシュートするリスクを最小限に抑えるのに役立つため

In [ ]:
# 訓練ループにコサイン減衰を反映させる
import math

min_lr = 0.1 * initial_lr
track_lrs = []
lr_increment = (peak_lr - initial_lr) / warmup_steps
global_step = -1

for epoch in range(n_epochs):
    for input_batch, target_batch in train_loader:
        optimizer.zero_grad()
        global_step += 1

        if global_step < warmup_steps: # 線形ウォームアップを適用
            lr = initial_lr + global_step * lr_increment
        else:
            # ウォームアップ後にコサイン減衰を適用
            progress = (
                (global_step - warmup_steps) /
                (total_training_steps - warmup_steps)
            )
            lr = (
                min_lr + (peak_lr - min_lr) * 0.5 * (1 + math.cos(math.pi * progress)) # 学習率をコサイン曲線の半周期で減衰させる
            )

        for param_group in optimizer.param_groups:
            param_group["lr"] = lr

        track_lrs.append(optimizer.param_groups[0]["lr"])



In [ ]:
# 学習曲線をプロット
plt.figure(figsize=(5, 3))
plt.ylabel("Learning rate")
plt.xlabel("Step")
plt.plot(range(total_training_steps), track_lrs)
plt.tight_layout()
plt.show()

- 学習率が線形ウォームフェーズから始まり、20ステップ後に最大値に到達するまで上昇する
- 20ステップの線形ウォームアップの後はコサイン減衰が始まり、学習率は極小値になるまで逓減してく

## D.3 勾配クリッピング
- 閾値を設定し、この閾値を超えた勾配は既定の最大値までダウンスケールさせる
- ダウンスケールさせることで、逆伝播中のモデルのパラメータ更新量が最適な範囲に確実に収まるようになる
- `clip_grad_norm()`関数の呼び出し時に`max_norm=1.0`を指定すると、勾配のノルムが1.0を超えなくなる
- 「ノルム」とは、勾配ベクトルの長さ（大きさ）の尺度のことであり、具体的にはユークリッドノルムとも呼ばれるL2ノルムを指す

In [ ]:
# 勾配クリッピングプロセスの使用

from previous_chapters import calc_loss_batch

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.to(device)
loss = calc_loss_batch(input_batch, target_batch, model, device)
loss.backward()


- `backward()`メソッドを呼び出すと、PyTorchが損失の勾配を計算し、各モデルの重み（パラメータ）テンソルの`grad`属性に格納

In [ ]:
# 最も大きい勾配を探すユーティリティ関数
def find_higest_gradient(model):
    max_grad = None
    for param in model.parameters():
        if param.grad is not None:
            grad_values = param.grad.data.flatten()
            max_grad_param = grad_values.max()
            if max_grad is None or max_grad_param > max_grad:
                max_grad = max_grad_param

    return max_grad

print(find_higest_gradient(model))

In [ ]:
# 勾配クリッピングを適用して、最も大きい勾配値にどのような影響を与えるか確認
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
print(find_higest_gradient(model))

- `max_norm=1.0`の勾配クリッピングを適用した後、最も大きい勾配値は勾配クリッピングを適用する前よりも大幅に小さくなる

## D.4 修正後の訓練関数
- 線形ウォームアップ
- コサイン減衰
- 勾配クリッピング

In [ ]:
from previous_chapters import evaluate_model, generate_and_print_sample

def train_model(model, train_loader, val_loader, optimizer, device,
                n_epochs, eval_freq, eval_iter, start_context, tokenizer,
                warmup_steps, initial_lr=3e-05, min_lr=1e-6):

    train_losses, val_losses, track_tokens_seen, track_lrs = [], [], [], []
    tokens_seen, global_step = 0, -1

    # オプティマイザから学習率の初期値を取得し、ピーク学習率として使用
    peak_lr = optimizer.param_groups[0]["lr"]
    # 訓練プロセスのイテレーションの総数を計算
    total_training_steps = len(train_loader) * n_epochs
    # ウォームアップフェーズの学習率の増分量を計算
    lr_increment = (peak_lr - initial_lr) / warmup_steps

    for epoch in range(n_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            global_step += 1

            if global_step < warmup_steps:
                lr = initial_lr + global_step * lr_increment
            else:
                progress = (
                    (global_step - warmup_steps) /
                    (total_training_steps - warmup_steps)
                )
                lr = (
                    min_lr + (peak_lr - min_lr) * 0.5 * (1 + math.cos(math.pi * progress))
                )

            # 計算した学習率をオプティマイザに適用
            for param_group in optimizer.param_groups:
                param_group["lr"] = lr

            track_lrs.append(lr)
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()

            if global_step > warmup_steps:
                # ウォームアップフェーズ後に勾配クリッピングを適用して勾配爆発を回避
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), max_norm=1.0
                )

            optimizer.step()
            tokens_seen += input_batch.numel()

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model,
                    train_loader,
                    val_loader,
                    device,
                    eval_iter
                )
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Iter {global_step:06d}):"
                    f"Training loss {train_loss:.3f}"
                    f"Validation loss {val_loss:.3f}")

        generate_and_print_sample(
             model, tokenizer, device, start_context
        )

    return train_losses, val_losses, track_tokens_seen, track_lrs

In [ ]:
# モデル訓練してみる
import tiktoken

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.to(device)
peak_lr = 5e-4
optimizer = torch.optim.AdamW(model.parameters(), weight_decay=0.1)
tokenizer = tiktoken.get_encoding("gpt2")

n_epochs = 15
train_losses, val_losses, tokens_seen, lrs = train_model(
    model, train_loader, val_loader, optimizer, device, n_epochs=n_epochs,
    eval_freq=5, eval_iter=1, start_context="Every effort moves you",
    tokenizer=tokenizer, warmup_steps=warmup_steps,
    initial_lr=1e-5, min_lr=1e-5
)

In [ ]:
# 学習率の曲線確認
plt.figure(figsize=(5, 3))
plt.plot(range(len(lrs)), lrs)
plt.ylabel("Learning rate")
plt.xlabel("Steps")
plt.show()

In [ ]:
# 損失のプロット
from previous_chapters import plot_losses

epochs_tensor = torch.linspace(1, n_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)
plt.tight_layout()
plt.savefig("3.pdf")
plt.show()